# Exploratory data analysis on single-cell SNV

Packages

In [367]:
from sklearn.decomposition import IncrementalPCA
from scipy.sparse import csr_matrix
from types import SimpleNamespace
import matplotlib.pyplot as plt
from pysam import VariantFile
from scipy.io import mmread
from pathlib import Path
import seaborn as sns
from umap import UMAP
import pandas as pd
import numpy as np
import os

Global variables

In [335]:
# class holding file paths
class PathFactory:
    def __init__(self, base):
        self._templates = {
                "path_alt_mtx": base / "outputs/VariantCalling/vartrix/{sample}/alt.mtx",
                "path_ref_mtx": base / "outputs/VariantCalling/vartrix/{sample}/ref.mtx",
                "path_barcodes": base / "outputs/Remapping/renamer/{sample}/barcodes.tsv", # column names of the matrices
                "path_snv_loci": base / "outputs/VariantCalling/vawk/{sample}.snv.loci.txt", # row names of the matrices
                "path_hom_ref": base / "outputs/VariantCalling-DNA/gatk_joint/{species}/hom_ref.vcf", # homozygous RNA-polymorphism loci
                "path_cell_identity": base / "outputs/notebooks/Seurat-integrate-Ptr-samples/{datetime}/cell_identity_all.csv",
                "path_dynamicviz_output": base / "outputs/notebooks/Seurat-integrate-Ptr-samples/{datetime}/dynamicviz.csv",
                "path_median_umap": base / "outputs/notebooks/Seurat-integrate-Ptr-samples/{datetime}/median_umap.csv",
                "path_variant_annotation": base / "tests/VariantAnnotation-ptr-multi/variant_annotation.tsv",
                "path_variant_location": base / "tests/VariantAnnotation-ptr-multi/variant_location.tsv",
                "path_annotation_synonym": base / "references/Ptr/annotation/Ptrichocarpa_533_v4.1.synonym.txt",
                "path_annotation_info": base / "references/Ptr/annotation/Ptrichocarpa_533_v4.1.annotation_info.txt",
                "path_seurat_data": base / "outputs/notebooks/Seurat-integrate-Ptr-samples/integrated_data.cca_ref=1_LogNorm_scaleFactor=10000.mtx"
        }

    @staticmethod
    def path_maker(template, **kwargs):
        template = str(template)
        for k, v in kwargs.items():
            template = template.replace("{"+ k +"}", str(v))
        return Path(template)

    def __getattr__(self, name):
        template = self._templates[name]
        return lambda **kwargs: PathFactory.path_maker(template, **kwargs)

workdir = Path("/home/woodydrylab/DiskArray/b05b01002/project_scRNAed")
paths = PathFactory(workdir)
path_output = workdir / "outputs/notebooks-new/EDA-on-scSNV/"
datetime = "20250729_172935" # SIMILARITY_THRESHOLD=0.7; integration=CCA; REF=ptr_tenx_batch1

In [347]:
"cluster_" + cell_identity["Cluster"].astype(str)

Barcode
AAACCTGAGCACCGCT-1_1     cluster_8
AAACCTGCAAGAGTCG-1_1     cluster_7
AAACCTGCACATTCGA-1_1     cluster_3
AAACCTGGTAAATGAC-1_1     cluster_3
AAACCTGGTCATCGGC-1_1     cluster_1
                           ...    
TTTGTTGGTACGATTC-1_15    cluster_1
TTTGTTGGTGCAGGAT-1_15    cluster_3
TTTGTTGGTGTAGGAC-1_15    cluster_2
TTTGTTGTCACTACTT-1_15    cluster_3
TTTGTTGTCCACTGAA-1_15    cluster_6
Name: Cluster, Length: 142621, dtype: object

In [348]:
# import/read paths
cell_identity = pd.read_csv(paths.path_cell_identity(datetime=datetime))
cell_identity["cluster_string"] = "cluster_" + cell_identity["Cluster"].astype(str)
cell_identity.index = cell_identity["Barcode"]

# create output directory
os.makedirs(path_output, exist_ok=True)

# REDITs scripts
path_REDIT_LLR = "src/REDITs/REDIT_LLR.R"
path_REDIT_regression = "src/REDITs/REDIT_regression.R"

# params
N_CELL_THRESHOLD = 100
COV_THRESHOLD = 10
PERCENT_DETECTED = 0.01
N_PCS = 30
B = 10
UMAP_PARAMS = {
    "n_components": 2,
    "n_neighbors": 15,
    "metric": "cosine",
    "min_dist": 0.1,
    "random_state": 42
}
DYNAMICVIZ_UMAP_PARAMS = {
    "n_neighbors": 15,
    "metric": "cosine",
    "min_dist": 0.1
}


# vars
color_palette = {
    1: "#1F77B4",
    2: "#7A4A33",
    3: "#E28413",
    4: "#F3EC5B",
    5: "#61C9A8",
    6: "#6362A7",
    7: "#FF0000",
    8: "#FF8F8F",
    9: "#D4D4D4",
    10: "#D4D4D4",
    11: "#D4D4D4",
    12: "#B03060"
}

# batch to name
batch_name = {
    "1": "ptr_tenx_batch1",
    "2": "ptr_tenx_batch2",
    "3": "ptr_tenx_tsv1",
    "4": "ptr_tenx_tsv2",
    "5": "ptr_tenx_tsv3",
    "6": "ptr_tenx_tsv4",
    "7": "ptr_tenx_tsv5",
    "8": "ptr_tenx_tst1",
    "9": "ptr_tenx_tst2",
    "10": "ptr_tenx_tst3",
    "11": "ptr_tenx_tst4",
    "12": "ptr_tenx_tso1",
    "13": "ptr_tenx_tso2",
    "14": "ptr_tenx_tso3",
    "15": "ptr_tenx_tso4"
}
name_batch = {v:k for k, v in batch_name.items()}
batch_color = {
    "1": "#F7FBFF",
    "2": "#DEEBF7",
    "3": "#C6DBEF",
    "4": "#9ECAE1",
    "5": "#6BAED6",
    "6": "#4292C6",
    "7": "#08519C",
    "8":  "#FEE5D9",
    "9":  "#FCAE91",
    "10": "#FB6A4A",
    "11": "#CB181D",
    "12":  "#E5F5E0",
    "13":  "#C7E9C0",
    "14":  "#A1D99B",
    "15":  "#74C476"
}

In [5]:
SKIPPED_BATCH = set(["1", "3", "8", "12"])

In [47]:
metadata = pd.DataFrame({
    "sample": [
        "ptr_tenx_batch2", "ptr_tenx_tsv2", "ptr_tenx_tsv3", "ptr_tenx_tsv4", "ptr_tenx_tsv5",
        "ptr_tenx_tst2", "ptr_tenx_tst3", "ptr_tenx_tst4",
        "ptr_tenx_tso2", "ptr_tenx_tso3", "ptr_tenx_tso4"
    ],
    "treatment": [
        "normal", "normal", "normal", "normal", "normal",
        "tension", "tension", "tension",
        "opposite", "opposite", "opposite"
    ],
    "individual": [
        "tung_2", "hsieh_2", "hsieh_3", "hsieh_4", "hsieh_5",
        "hsieh_7", "hsieh_8", "hsieh_9",
        "hsieh_7", "hsieh_8", "hsieh_9"
    ]
})
metadata.index = metadata["sample"]
metadata.head()

,sample,treatment,individual
0,ptr_tenx_batch2,normal,tung_2
1,ptr_tenx_tsv2,normal,hsieh_2
2,ptr_tenx_tsv3,normal,hsieh_3
3,ptr_tenx_tsv4,normal,hsieh_4
4,ptr_tenx_tsv5,normal,hsieh_5


Class for handling sparse matrices

In [115]:
class SparseMatrices:
    def __init__(
        self,
        alt_mtx, 
        ref_mtx,
        index:list,
        columns:list
    ):  
        # checks        
        if alt_mtx.shape != (len(index), len(columns)):
            raise ValueError("Problem with index and columns")
        
        self.alt_mtx = alt_mtx.tocsr()
        self.ref_mtx = ref_mtx.tocsr()
        self.index = {name: idx for idx, name in enumerate(index)}
        self.columns = {name: idx for idx, name in enumerate(columns)}
        self._index = index
        self._columns = columns
        self.cov_mtx = alt_mtx + ref_mtx
        self.alt_frac = alt_mtx.multiply(self.cov_mtx.power(-1)).tocsr()
           
    def mask_alt_frac(self, cov_thresh:int):
        """
        mask alt_frac where coverage < cov_thresh
        """
        masks = self.cov_mtx < cov_thresh
        self.alt_frac = self.alt_frac.multiply(masks.astype(float))

    def shape(self):
        return (self.index.__len__(), self.columns.__len__())

    def __repr__(self):
        return (
            f"Sparse matrix of {self.shape()[0]:,} loci x {self.shape()[1]:,} cell"
        )
        
    def subset_matrix(attr_name):
        def decorator(func):
            def wrapper(self, index=None, columns=None):
                matrix = getattr(self, attr_name)

                # Row indexing
                if index is None:
                    row_idx = slice(None)
                else:
                    if not isinstance(index, list):
                        index = list(index)
                    if isinstance(index[0], bool):
                        row_idx = index
                    elif isinstance(index[0], str):
                        row_idx = [self.index[i] for i in index]

                # Column indexing
                if columns is None:
                    col_idx = slice(None)
                else:
                    if not isinstance(columns, list):
                        columns = list(columns)
                    if isinstance(columns[0], bool):
                        col_idx = columns
                    elif isinstance(columns[0], str):
                        col_idx = [self.columns[c] for c in columns]

                return matrix[row_idx, :][:, col_idx]

            return wrapper
        return decorator

    @subset_matrix("alt_frac")
    def subset_alt_frac(self): pass

    @subset_matrix("cov_mtx")
    def subset_cov_mtx(self): pass

    @subset_matrix("alt_mtx")
    def subset_alt_mtx(self): pass

    @subset_matrix("ref_mtx")
    def subset_ref_mtx(self): pass


#### Exploratory Analysis

Get homozygous loci

In [9]:
hom_ref = VariantFile(paths.path_hom_ref(species="ptr"))
hom_loci = [f"{i.chrom}:{i.pos}" for i in hom_ref.fetch()]
hom_loci[:5]

['Chr01:10586', 'Chr01:15129', 'Chr01:15250', 'Chr01:15273', 'Chr01:15274']

In [235]:
# read matrices
matrix_dict = {}
for sample in metadata["sample"].unique():
    batch = name_batch[sample]
    sparse_matrices = SparseMatrices(
        alt_mtx=mmread(paths.path_alt_mtx(species="ptr", sample=sample)),
        ref_mtx=mmread(paths.path_ref_mtx(species="ptr", sample=sample)),
        index=[i.strip() for i in open(paths.path_snv_loci(species="ptr", sample=sample), "r")],
        columns=[f"{i.strip()}_{batch}" for i in open(paths.path_barcodes(species="ptr", sample=sample), "r")]
    )
    matrix_dict[sample] = sparse_matrices

In [236]:
matrix_dict

{'ptr_tenx_batch2': Sparce matrix of 334,423 loci x 13,968 cell,
 'ptr_tenx_tsv2': Sparce matrix of 313,885 loci x 14,972 cell,
 'ptr_tenx_tsv3': Sparce matrix of 310,362 loci x 12,412 cell,
 'ptr_tenx_tsv4': Sparce matrix of 278,213 loci x 9,070 cell,
 'ptr_tenx_tsv5': Sparce matrix of 241,215 loci x 11,413 cell,
 'ptr_tenx_tst2': Sparce matrix of 293,299 loci x 13,715 cell,
 'ptr_tenx_tst3': Sparce matrix of 311,147 loci x 12,065 cell,
 'ptr_tenx_tst4': Sparce matrix of 309,442 loci x 10,637 cell,
 'ptr_tenx_tso2': Sparce matrix of 348,060 loci x 11,060 cell,
 'ptr_tenx_tso3': Sparce matrix of 345,776 loci x 8,416 cell,
 'ptr_tenx_tso4': Sparce matrix of 306,139 loci x 9,166 cell}

Narrow down to loci where at least one cell in one sample showed coverage > `COV_THRESHOLD`

In [379]:
loci_to_test = set()
loci_union = set()
for sample in metadata["sample"].unique():
    # simple union
    loci_union = loci_union.union(set(matrix_dict[sample]._index))

    # any > COV_THRESHOLD
    loci_any_cell_gt_thresh = (matrix_dict[sample].cov_mtx.max(1) >= COV_THRESHOLD).toarray().flatten().tolist()
    loci_tmp = np.array(matrix_dict[sample]._index)[loci_any_cell_gt_thresh]
    loci_to_test = loci_to_test.union(loci_tmp.tolist())

loci_to_test = list(loci_to_test)
loci_to_test.sort()
len(loci_to_test)

19769

In [386]:
hom_loci_to_test = list(set(loci_to_test) & set(hom_loci))
print(
    f"RNA polymorphism loci: {len(loci_union):,}\n"
    f"Hom loci: {len(hom_loci):,}\n"
    f"RNA editing sites: {len(set(hom_loci) & loci_union):,}\n"
    f"RNA editing sites (not all-sample-all-cell coverage < {COV_THRESHOLD}): {len(hom_loci_to_test)}"
)

RNA polymorphism loci: 1,147,751
Hom loci: 438,550
RNA editing sites: 366,332
RNA editing sites (not all-sample-all-cell coverage < 10): 1748


In [372]:
cells_per_loci = []

for locus in hom_loci_to_test:
    path_output_locus = path_output / locus
    # get alt and ref count if locus in index
    sdata_list = []
    smetadata_list = []
    for sample in metadata["sample"].unique():
        if locus in matrix_dict[sample].index:
            sdata = np.vstack([
                matrix_dict[sample].subset_alt_mtx(index=[locus], columns=None).toarray(),
                matrix_dict[sample].subset_ref_mtx(index=[locus], columns=None).toarray()
            ])
            sdata = pd.DataFrame(sdata)
            sdata.index = ["alt", "ref"]
            sdata.columns = matrix_dict[sample]._columns
            sdata = sdata.loc[:, sdata.sum(0) > COV_THRESHOLD]
            if sdata.shape[1] > 0:
                smetadata = pd.DataFrame({
                    "cluster": cell_identity.loc[sdata.columns, "cluster_string"].values,
                    "treatment": metadata.loc[sample, "treatment"],
                    "individual": metadata.loc[sample, "individual"]
                })
                smetadata.index = sdata.columns
                sdata_list.append(sdata)
                smetadata_list.append(smetadata)
    
    if len(sdata_list) > 0:
        ldata = pd.concat(sdata_list, axis=1)
        lmetadata = pd.concat(smetadata_list, axis=0)
        dout = pd.concat([ldata.T, lmetadata], axis=1)
        if ldata.shape[1] > N_CELL_THRESHOLD:
            os.makedirs(path_output_locus, exist_ok=True)
            dout.to_csv(path_output_locus / f"data.csv")

In [374]:
os.listdir(path_output).__len__()

211